# M1  Comparativa Final: Pandas vs Polars vs Dask vs cuDF
## Sistema de Recomendación Paralelo para E-Commerce  RetailRocket Dataset
### Entrega 3  Análisis de resultados

**Corre en Google Colab con GPU activada.**




## 0. Configuración inicial

In [1]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))


ModuleNotFoundError: No module named 'torch'

### Instalar cuDF (RAPIDS)

Usa el paquete pip oficial de NVIDIA para Colab (más liviano que el
instalador conda tradicional de RAPIDS)


In [ ]:
!pip install --extra-index-url=https://pypi.nvidia.com cudf-cu12


### Traer el código del repositorio

In [ ]:
!git clone https://github.com/DavidMoraV/Proyecto-Paralela_E-commerce.git
%cd Proyecto-Paralela_E-commerce/Codigos
!pip install -q polars "dask[dataframe]" psutil


### Subir `events.csv`

Sube el archivo original completo  es el que se usa para
medir ingesta y limpieza desde cero.


In [ ]:
from google.colab import files
from pathlib import Path

Path("data_raw").mkdir(exist_ok=True)
print("Sube events.csv:")
subido = files.upload()
import shutil
shutil.move(list(subido.keys())[0], "data_raw/events.csv")


## 1. Funciones de carga, limpieza y transformación por herramienta

Cada función se divide en 3 pasos medidos por separado: **carga** (leer el
CSV), **limpieza** (deduplicar, quitar nulos, filtrar eventos válidos), y
**transformación** (parsear timestamp, calcular `peso_implicito`).


In [ ]:
import sys
sys.path.append("src")
from pipeline_datos import detectar_separador, resolver_ruta

RUTA_EVENTOS = Path("data_raw/events.csv")
PESOS = {"view": 1, "addtocart": 3, "transaction": 5}


In [ ]:
# Pandas
import pandas as pd

def pandas_carga(ruta):
    sep = detectar_separador(ruta)
    return pd.read_csv(ruta, sep=sep, dtype={"itemid": "float64"})

def pandas_limpieza(df):
    df = df.drop_duplicates()
    df = df.dropna(subset=["itemid", "visitorid"])
    return df[df["event"].isin(["view", "addtocart", "transaction"])]

def pandas_transformacion(df):
    df = df.copy()
    df["fecha"] = pd.to_datetime(df["timestamp"], unit="ms")
    df["peso_implicito"] = df["event"].map(PESOS).fillna(1).astype(int)
    df["hora_del_dia"] = df["fecha"].dt.hour
    return df


In [ ]:
# Polars
import polars as pl

def polars_carga(ruta):
    sep = detectar_separador(ruta)
    lf = pl.scan_csv(ruta, separator=sep, schema_overrides={"itemid": pl.Float64, "transactionid": pl.Float64})
    if sep == ";":
        lf = lf.with_columns(pl.col("timestamp").str.replace(",", ".").cast(pl.Float64).cast(pl.Int64))
    else:
        lf = lf.with_columns(pl.col("timestamp").cast(pl.Int64))
    return lf.collect()

def polars_limpieza(df):
    return (
        df.unique()
        .filter(pl.col("itemid").is_not_null() & pl.col("visitorid").is_not_null())
        .filter(pl.col("event").is_in(["view", "addtocart", "transaction"]))
    )

def polars_transformacion(df):
    return df.with_columns([
        pl.from_epoch("timestamp", time_unit="ms").alias("fecha"),
        pl.col("event").replace_strict(PESOS, default=1, return_dtype=pl.Int32).alias("peso_implicito"),
    ]).with_columns(pl.col("fecha").dt.hour().alias("hora_del_dia"))


In [ ]:
#  Dask
import dask.dataframe as dd

def dask_carga(ruta):
    sep = detectar_separador(ruta)
    df = dd.read_csv(ruta, sep=sep, dtype={"itemid": "float64"}, blocksize="16MB")
    # .persist() fuerza el computo de esta etapa AHORA (a diferencia de
    # .compute(), no la convierte a pandas, el resultado sigue siendo una
    # coleccion Dask, lista para que la siguiente etapa construya sobre un
    # resultado YA materializado, no sobre el grafo completo desde cero).
    return df.persist()

def dask_limpieza(df):
    df = df.drop_duplicates()
    df = df.dropna(subset=["itemid", "visitorid"])
    df = df[df["event"].isin(["view", "addtocart", "transaction"])]
    return df.persist()

def dask_transformacion(df):
    df = df.copy()
    df["fecha"] = dd.to_datetime(df["timestamp"], unit="ms")
    df["peso_implicito"] = df["event"].map(PESOS, meta=("peso_implicito", "int64"))
    df["hora_del_dia"] = df["fecha"].dt.hour
    return df.compute()  # aqui si conviene el resultado final como pandas


In [ ]:
# cuDF (GPU)
# API deliberadamente muy similar a pandas si algo falla aqui, suele ser
# por una diferencia de tipos (cuDF es mas estricto con nulos/tipos mixtos
# que pandas) o por la version especifica de cuDF instalada.
import cudf

def cudf_carga(ruta):
    sep = detectar_separador(ruta)
    return cudf.read_csv(ruta, sep=sep)

def cudf_limpieza(df):
    df = df.drop_duplicates()
    df = df.dropna(subset=["itemid", "visitorid"])
    return df[df["event"].isin(["view", "addtocart", "transaction"])]

def cudf_transformacion(df):
    df = df.copy()
    df["fecha"] = cudf.to_datetime(df["timestamp"], unit="ms")
    df["peso_implicito"] = df["event"].map(PESOS).fillna(1).astype("int32")
    df["hora_del_dia"] = df["fecha"].dt.hour
    return df


## 2. Benchmark: carga, limpieza y transformación, por separado

Con warm-up + repeticiones, igual que el resto del proyecto.


In [ ]:
import time
import numpy as np

def medir_etapas(nombre, fn_carga, fn_limpieza, fn_transformacion, n_repeticiones=3, warm_up=1):
    resultados = {}
    for etapa, fn, entrada_previa in [
        ("carga", fn_carga, None),
        ("limpieza", fn_limpieza, "carga"),
        ("transformacion", fn_transformacion, "limpieza"),
    ]:
        # warm-up
        entrada = resultados.get(f"_df_{entrada_previa}") if entrada_previa else RUTA_EVENTOS
        for _ in range(warm_up):
            _ = fn(entrada)

        tiempos = []
        salida = None
        for _ in range(n_repeticiones):
            t0 = time.perf_counter()
            salida = fn(entrada)
            tiempos.append(time.perf_counter() - t0)

        resultados[etapa] = {"media_s": float(np.mean(tiempos)), "std_s": float(np.std(tiempos))}
        resultados[f"_df_{etapa}"] = salida  # se pasa a la siguiente etapa
        print(f"  {nombre} - {etapa}: {np.mean(tiempos):.3f}s (+/- {np.std(tiempos):.3f}s)")

    return {k: v for k, v in resultados.items() if not k.startswith("_df_")}


print("=== Pandas ===")
r_pandas = medir_etapas("Pandas", pandas_carga, pandas_limpieza, pandas_transformacion)

print("=== Polars ===")
r_polars = medir_etapas("Polars", polars_carga, polars_limpieza, polars_transformacion)

print("=== Dask ===")
r_dask = medir_etapas("Dask", dask_carga, dask_limpieza, dask_transformacion)

print("=== cuDF (GPU) ===")
r_cudf = medir_etapas("cuDF", cudf_carga, cudf_limpieza, cudf_transformacion)


## 3. Tabla comparativa final

In [ ]:
import pandas as pd

filas = []
for nombre, r in [("Pandas", r_pandas), ("Polars", r_polars), ("Dask", r_dask), ("cuDF (GPU)", r_cudf)]:
    tiempo_total = sum(r[etapa]["media_s"] for etapa in ["carga", "limpieza", "transformacion"])
    filas.append({
        "herramienta": nombre,
        "carga_s": r["carga"]["media_s"],
        "limpieza_s": r["limpieza"]["media_s"],
        "transformacion_s": r["transformacion"]["media_s"],
        "total_s": tiempo_total,
    })

df_comparativa = pd.DataFrame(filas)
tiempo_base = df_comparativa.loc[df_comparativa["herramienta"] == "Pandas", "total_s"].iloc[0]
df_comparativa["speedup"] = (tiempo_base / df_comparativa["total_s"]).round(2)

# "Eficiencia" aqui se define como speedup / recursos paralelos usados, para
# mantener el mismo marco conceptual que el resto del proyecto (Amdahl,
# workers de Dask). Pandas=1 nucleo, Polars=todos los nucleos disponibles,
# Dask=4 workers (configuracion usada en toda la Entrega 2/3), cuDF=1 GPU.
import os
n_cores = os.cpu_count()
recursos = {"Pandas": 1, "Polars": n_cores, "Dask": 4, "cuDF (GPU)": 1}
df_comparativa["recursos_usados"] = df_comparativa["herramienta"].map(recursos)
df_comparativa["eficiencia"] = (df_comparativa["speedup"] / df_comparativa["recursos_usados"]).round(3)

Path("resultados/entrega3").mkdir(parents=True, exist_ok=True)
df_comparativa.to_csv("resultados/entrega3/comparativa_final_pandas_polars_dask_cudf.csv", index=False)
df_comparativa


## 5. Descargar resultados

In [ ]:
from google.colab import files
files.download("resultados/entrega3/comparativa_final_pandas_polars_dask_cudf.csv")
